# Day3_03A_CrewAI_First_Crew

## CrewAI - Building Your First Crew

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Session:** Day 3 - CrewAI Introduction  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand why CrewAI is useful after learning OpenAI Agents SDK.
- Explain the four core CrewAI concepts: **Agent, Task, Tool, Crew**.
- Create role-based Agents using role, goal, and backstory.
- Create Tasks with clear expected outputs.
- Run a simple Crew using a sequential process.
- Relate CrewAI to DSU-style teaching and enterprise workflows.

---

## Teaching Context

In the previous section, we built Agents using the OpenAI Agents SDK.

We learned:

- Agents
- Tools
- Handoffs
- Parallel execution
- Quality gates
- Context-aware workflows

Now we move to CrewAI.

CrewAI helps us organize multiple AI Agents as a **team**.

Instead of writing all orchestration logic manually, we describe:

- Who the Agents are
- What each Agent should do
- What tasks must be completed
- How the Crew should work together


# 1. Why CrewAI?

Let us start with a simple question:

> If one Agent can answer questions, why do we need a Crew?

Because real work is rarely done by one person.

In a college FDP preparation example:

- One person researches the topic.
- One person prepares the lecture module.
- One person reviews the material.
- One person improves the final output.

CrewAI helps us model this kind of teamwork.

---

## Single Agent Limitation

A single Agent may:

- Try to do everything alone
- Miss specialist viewpoints
- Produce shallow outputs
- Struggle with complex workflows

---

## CrewAI Advantage

CrewAI lets us create:

- Multiple expert Agents
- Role-based collaboration
- Task chaining
- Clear outputs
- Scalable workflows


# 2. The Four Core CrewAI Concepts

CrewAI is built around four simple concepts.

| Concept | Meaning | Simple Question |
|---|---|---|
| Agent | Who does the work? | Who is the specialist? |
| Task | What needs to be done? | What is the assignment? |
| Tool | What can they use? | What external help is available? |
| Crew | How do they collaborate? | How does the team work together? |

---

## Mental Model

```text
Agent + Task + Tool + Crew = Collaborative AI Workflow
```

For this first notebook, we will focus mainly on:

- Agent
- Task
- Crew

We will introduce Tools in a later CrewAI notebook.


# 3. CrewAI Architecture for Our First Demo

We will build a simple DSU teaching-content crew.

```text
Faculty Topic
   ↓
Researcher Agent
   ↓
Writer Agent
   ↓
Reviewer Agent
   ↓
Final Teaching Brief
```

The final output will be a short teaching brief that can be used by faculty.

---

## Why this example?

It is:

- Classroom friendly
- Easy to understand
- Relevant to faculty
- Similar to real academic preparation
- Extendable later with tools and context


# 4. Setup

We will use CrewAI.

Before running this notebook, make sure your environment has:

```bash
pip install crewai crewai-tools python-dotenv
```

Also make sure your project root contains the `.env` file with your API key.

Example:

```text
FDP-LLM/
  .env
  requirements.txt
  day3-openai-crewai/
    Day3_03A_CrewAI_First_Crew.ipynb
```


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from crewai import Agent, Task, Crew, Process

# 5. Load Environment Variables

This is the same root `.env` loading pattern we used in earlier notebooks.

It works even if this notebook is inside a day-wise folder.


In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 6. Create the First Agent - Researcher

A CrewAI Agent usually has three important fields:

- **role** - who the Agent is
- **goal** - what the Agent wants to achieve
- **backstory** - why the Agent is good at the job

This makes CrewAI very beginner-friendly because the Agent is described like a real team member.


In [ ]:
researcher = Agent(
    role="AI Education Research Specialist",
    goal="Find simple and relevant points about Agentic AI for engineering faculty",
    backstory="""
    You are an AI education researcher with experience in explaining emerging
    technologies to engineering faculty. You focus on clarity, practical relevance,
    and examples suitable for classroom teaching.
    """,
    verbose=True,
    allow_delegation=False,
)

## What did we just learn?

We created an Agent, but not by writing a long system prompt.

Instead, we described:

```text
Role + Goal + Backstory
```

This is one of the biggest strengths of CrewAI.

It makes Agents easier to explain to non-developers.


# 7. Create the Second Agent - Writer

The Writer Agent will convert research points into a short teaching module.


In [ ]:
writer = Agent(
    role="Teaching Content Writer",
    goal="Convert research points into a short, structured teaching brief",
    backstory="""
    You are a faculty-friendly content writer. You convert technical material
    into simple teaching notes, examples, and classroom explanations.
    You avoid jargon and keep the content practical.
    """,
    verbose=True,
    allow_delegation=False,
)

# 8. Create the Third Agent - Reviewer

The Reviewer Agent will check whether the teaching brief is clear and useful.


In [ ]:
reviewer = Agent(
    role="Academic Quality Reviewer",
    goal="Review the teaching brief for clarity, completeness, and classroom usefulness",
    backstory="""
    You are a senior academic reviewer. You check whether teaching material
    is clear, practical, and suitable for engineering faculty with minimal coding background.
    You suggest improvements where needed.
    """,
    verbose=True,
    allow_delegation=False,
)

# Pause & Reflect

We now have three Agents.

| Agent | Role |
|---|---|
| Researcher | Finds useful points |
| Writer | Creates teaching brief |
| Reviewer | Checks quality |

This is similar to how real academic content gets prepared.


# 9. Create the First Task - Research Task

A CrewAI Task describes:

- What needs to be done
- Which Agent should do it
- What output is expected

Clear task descriptions improve output quality.


In [ ]:
research_task = Task(
    description="""
    Research the topic: Agentic AI for engineering education.

    Identify:
    1. What Agentic AI means
    2. Why it matters for engineering faculty
    3. One simple classroom example
    4. One enterprise example
    5. Common misconceptions beginners may have
    """,
    expected_output="A concise research brief with 5 clear sections.",
    agent=researcher,
)

# 10. Create the Second Task - Writing Task

This task uses the research output and creates a teaching brief.

In CrewAI, task outputs can naturally flow into later tasks when using sequential process.


In [ ]:
write_task = Task(
    description="""
    Using the research brief, create a short teaching module for faculty.

    The module should include:
    1. A simple definition
    2. A DSU classroom example
    3. A short explanation of why Agents are different from chatbots
    4. Three teaching points faculty can use in class
    """,
    expected_output="A 400-word teaching module written in simple faculty-friendly language.",
    agent=writer,
    context=[research_task],
)

# 11. Create the Third Task - Review Task

The Reviewer will evaluate the teaching module and improve it.

This introduces the idea of a **quality loop**, which we already learned in OpenAI SDK.


In [ ]:
review_task = Task(
    description="""
    Review the teaching module.

    Check:
    1. Is it simple enough for engineering faculty?
    2. Does it include a classroom example?
    3. Does it avoid unnecessary jargon?
    4. Is the content useful for a 10-minute explanation?

    Provide an improved final version.
    """,
    expected_output="An improved final teaching brief ready for classroom use.",
    agent=reviewer,
    context=[write_task],
)

# 12. Create the Crew

Now we combine:

- Agents
- Tasks
- Process

The process tells CrewAI how tasks should run.

For this first demo, we use:

```python
Process.sequential
```

This means tasks run one after another.


In [ ]:
teaching_crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[research_task, write_task, review_task],
    process=Process.sequential,
    verbose=True,
)

# 13. Run the Crew

Now let us run our first CrewAI workflow.

This is called `kickoff()`.


In [ ]:
result = teaching_crew.kickoff()

print(result)

# 14. What Did We Just Build?

We built a simple multi-agent workflow.

```text
Topic
   ↓
Researcher
   ↓
Writer
   ↓
Reviewer
   ↓
Final Teaching Brief
```

This is very similar to a real-world academic preparation process.

The difference is that each specialist is represented by an AI Agent.


# 15. CrewAI vs OpenAI SDK in This Example

| Question | OpenAI SDK | CrewAI |
|---|---|---|
| How do we define Agents? | Instructions | Role, Goal, Backstory |
| How do we define work? | Python orchestration | Tasks |
| How do outputs flow? | Manually coded | Task context |
| How do Agents collaborate? | You design flow | Crew manages flow |
| Beginner friendliness | Moderate | Easier |

---

## Teaching Point

CrewAI does not remove the need to understand Agents.

It makes Agent collaboration easier to describe and build.


# 16. Modify the Topic

Now let us make this interactive.

Change the topic below and run the crew again.

Try topics like:

- RAG for engineering education
- Prompt engineering for faculty
- AI ethics in classrooms
- Agentic AI for software testing


In [ ]:
# Classroom modification cell

research_task.description = """
Research the topic: RAG for engineering education.

Identify:
1. What RAG means
2. Why it matters for engineering faculty
3. One simple classroom example
4. One enterprise example
5. Common misconceptions beginners may have
"""

result = teaching_crew.kickoff()

print(result)

# 17. Classroom Exercise

Create a new Crew for:

> Preparing a 15-minute lecture on Prompt Engineering

Use three Agents:

1. Researcher
2. Writer
3. Reviewer

Expected final output:

- Simple definition
- Classroom example
- One activity
- Key takeaways


In [ ]:
# Exercise Starter Code

# TODO:
# 1. Create a prompt engineering researcher Agent
# 2. Create a prompt engineering writer Agent
# 3. Create a prompt engineering reviewer Agent
# 4. Create three tasks
# 5. Create a Crew
# 6. Run kickoff()



# 18. Challenge Exercise

Add one more Agent:

## Quiz Creator

The Quiz Creator should create:

- 3 MCQs
- 2 short-answer questions
- Answer key

Then add the quiz task after the teaching module task.

This will extend the Crew from 3 Agents to 4 Agents.


# 19. Common Mistakes

Avoid these mistakes while building CrewAI workflows:

1. Giving vague Agent roles.
2. Writing goals that are too broad.
3. Forgetting `expected_output`.
4. Creating tasks without clear sequence.
5. Using too many Agents for a simple problem.
6. Making backstories too long and confusing.
7. Forgetting that task context controls chaining.


# 20. Key Takeaways

In this notebook, we learned:

- CrewAI helps create collaborative Agent teams.
- The four key CrewAI concepts are Agent, Task, Tool, and Crew.
- Agents are defined using role, goal, and backstory.
- Tasks define what each Agent must produce.
- Context allows one task to use the output of previous tasks.
- A Crew brings Agents and Tasks together.
- `kickoff()` runs the Crew.

---

## Final Mental Model

```text
Agent = Who does the work
Task = What needs to be done
Tool = What the Agent can use
Crew = How Agents collaborate
```

You have now built your first CrewAI workflow.
